In [ ]:
'''
Ken Jinks 2026
This file contains code to produce string art from an image.
An array of 2D points represent the nails.
An average of pixel values over a line between two points gives the probability that a string should be placed. 
This is a stochatic tomographic approach.
'''

In [1]:
import numpy as np
from PIL import Image

In [4]:
"""
June 2026 I asked Claude AI to produce this alogorithm. This is the prompt I used:

###I'm looking for a speedy python algorithm, possibly using the libraries pillow and numpy. 
The method will take the parameters x1, y1, x2, y2, an image and and return a colour. 
I want the method to sum all the pixels from the image in a straight line from (x1, y1) to (x2, y2) 
and return a normalized colour by dividing the sum by the number of pixels.###
"""
def sample_line_colour(x1: int, y1: int, x2: int, y2: int, image: Image.Image) -> tuple[int, int, int]:
    """
    Samples all pixels along a straight line from (x1, y1) to (x2, y2) in the
    given PIL image and returns their average as an RGB tuple.

    Parameters:
        x1, y1: Start point (column, row)
        x2, y2: End point (column, row)
        image:  A PIL Image object (any mode; converted to RGB internally)

    Returns:
        A tuple (R, G, B) with values in [0, 255].
    """
    img_array = np.asarray(image.convert("RGB"))  # H x W x 3, uint8

    num_pixels = max(abs(x2 - x1), abs(y2 - y1)) + 1

    # Bresenham-style pixel coordinates via linspace (vectorised, no Python loop)
    xs = np.round(np.linspace(x1, x2, num_pixels)).astype(np.intp)
    ys = np.round(np.linspace(y1, y2, num_pixels)).astype(np.intp)

    # Clamp to image bounds so out-of-range endpoints don't raise an IndexError
    h, w = img_array.shape[:2]
    xs = np.clip(xs, 0, w - 1)
    ys = np.clip(ys, 0, h - 1)

    # Fancy-index all pixels at once, then average (stays in float until the end)
    pixels = img_array[ys, xs]          # shape: (num_pixels, 3)
    avg = pixels.mean(axis=0)           # shape: (3,)

    return tuple(np.round(avg).astype(np.uint8).tolist())

In [3]:
# --- quick smoke-test ---
if __name__ == "__main__":
    # Solid red 100x100 image → should return (255, 0, 0)
    red = Image.new("RGB", (100, 100), (255, 0, 0))
    print(sample_line_colour(0, 0, 99, 99, red))   # (255, 0, 0)

    # Gradient: left half blue, right half green → diagonal avg ≈ (0, 127, 127)
    gradient = Image.new("RGB", (100, 100), (0, 0, 255))
    right_half = Image.new("RGB", (50, 100), (0, 255, 0))
    gradient.paste(right_half, (50, 0))
    print(sample_line_colour(0, 50, 99, 50, gradient))  # ≈ (0, 127, 127)

(255, 0, 0)
(0, 128, 128)
